# audio_controls

> Audio playback controls for the alignment card stack: auto-navigate toggle

In [ ]:
#| default_exp components.audio_controls

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, Span, Input, Label

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_input.toggle import toggle, toggle_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, bg_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.utilities.typography import font_size
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

# Web Audio library — shared speed selector component + config
from cjm_fasthtml_web_audio.models import WebAudioConfig
from cjm_fasthtml_web_audio.components import render_speed_selector

## HTML IDs

HTML ID constants for alignment audio control elements.

In [ ]:
#| export
class AlignAudioControlIds:
    """HTML ID constants for alignment audio control elements."""

    AUTO_NAV_TOGGLE = "sd-align-auto-nav-toggle"
    AUDIO_CONTROLS = "sd-align-audio-controls"

## Speed Selector Config

The speed selector itself is provided by `cjm_fasthtml_web_audio.components.render_speed_selector`. We only need a minimal `WebAudioConfig` here (namespace + `enable_speed=True`) to drive its namespace-derived IDs and `window.setAlignSpeed` / `window.cycleAlignSpeed*` wiring.

Defined locally (rather than reusing `ALIGN_AUDIO_CONFIG` from `components.callbacks`) to avoid a circular import — `callbacks` imports symbols from this module.

In [ ]:
#| export
# Minimal config used only to drive the shared render_speed_selector.
# Must stay namespace-compatible with ALIGN_AUDIO_CONFIG (defined in components.callbacks)
# so both configs resolve to the same `window.setAlignSpeed`, `sd-align-speed-select`, etc.
_ALIGN_SPEED_CONFIG = WebAudioConfig(
    namespace="align",
    indicator_selector="",
    enable_speed=True,
)

## Auto-Navigate Toggle

Toggle switch to enable automatic navigation to the next VAD chunk
when audio playback of the current chunk completes.

In [ ]:
#| export
# CSS classes for toggle state coloring
_TOGGLE_BG_OFF = str(bg_dui.error)    # Red when auto-play disabled
_TOGGLE_BG_ON = str(bg_dui.success)   # Green when auto-play enabled

def _toggle_color_js(toggle_id:str) -> str:  # JS snippet to sync toggle color classes
    """Generate JS to swap bg-error/bg-success on the toggle based on checked state."""
    return (
        f"var _t=document.getElementById('{toggle_id}');"
        f"if(_t){{_t.classList.remove('{_TOGGLE_BG_OFF}','{_TOGGLE_BG_ON}');"
        f"_t.classList.add(_t.checked?'{_TOGGLE_BG_ON}':'{_TOGGLE_BG_OFF}');}}"
    )

def render_align_auto_navigate_toggle(
    enabled:bool=False,  # Whether auto-navigate is enabled
) -> Any:  # Auto-navigate toggle component
    """Render auto-navigate toggle switch for alignment audio (client-side only)."""
    toggle_id = AlignAudioControlIds.AUTO_NAV_TOGGLE
    color_js = _toggle_color_js(toggle_id)
    onchange_js = f"if(window.setAlignAutoNavigate) window.setAlignAutoNavigate(this.checked);{color_js}"

    # Only pass checked when enabled (FastHTML renders checked=False as an attribute)
    check_attr = {"checked": True} if enabled else {}
    color_cls = _TOGGLE_BG_ON if enabled else _TOGGLE_BG_OFF

    return Div(
        Label(
            Span(
                "Auto-play:",
                cls=combine_classes(font_size.sm, text_tiers.secondary, m.r(2))
            ),
            Input(
                type="checkbox",
                id=toggle_id,
                name="auto_navigate",
                cls=combine_classes(toggle, toggle_sizes.sm, color_cls),
                onchange=onchange_js,
                **check_attr,
            ),
            cls=combine_classes(flex_display, items.center)
        ),
        cls=combine_classes(flex_display, items.center)
    )

## Audio Controls Container

Wraps the toggle in an OOB-targetable container for chrome switching.

In [ ]:
#| export
def render_align_audio_controls(
    current_speed:float=1.0,  # Current playback speed
    auto_navigate:bool=False,  # Whether auto-navigate is enabled
    speed_url:str="",  # URL for speed changes (server persistence)
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Combined audio controls component
    """Render alignment audio controls (speed selector + auto-navigate toggle)."""
    return Div(
        render_speed_selector(_ALIGN_SPEED_CONFIG, current_speed, speed_url),
        render_align_auto_navigate_toggle(auto_navigate),
        id=AlignAudioControlIds.AUDIO_CONTROLS,
        cls=combine_classes(flex_display, items.center, gap(4)),
        hx_swap_oob="true" if oob else None,
    )

## Tests

In [ ]:
from fasthtml.common import to_xml
import re

def _get_toggle_classes(html):
    """Extract class attribute from the toggle input element."""
    m = re.search(r'id="sd-align-auto-nav-toggle"[^>]*class="([^"]*)"', html)
    return m.group(1) if m else ""

# Test enabled toggle — green background, checked
html = to_xml(render_align_auto_navigate_toggle(enabled=True))
assert ' checked' in html
assert 'sd-align-auto-nav-toggle' in html
assert 'setAlignAutoNavigate' in html
assert 'hx-post' not in html  # No server persistence
toggle_cls = _get_toggle_classes(html)
assert 'bg-success' in toggle_cls
assert 'bg-error' not in toggle_cls

# Test disabled state — red background, no checked
html_off = to_xml(render_align_auto_navigate_toggle(enabled=False))
assert ' checked' not in html_off
toggle_cls_off = _get_toggle_classes(html_off)
assert 'bg-error' in toggle_cls_off
assert 'bg-success' not in toggle_cls_off

# Test color JS is in onchange (classList swap logic)
assert 'classList.remove' in html
assert 'classList.add' in html

# Container delegates speed selector to cjm-fasthtml-web-audio's render_speed_selector.
# Verify the container wires the library correctly at the integration boundary:
# namespace-derived IDs/functions present, persisted speed flows through, OOB attr applied.

# Default speed + no change_url: select rendered, no HTMX persistence, no sync script
html_default = to_xml(render_align_audio_controls(current_speed=1.0, auto_navigate=False))
assert 'sd-align-speed-select' in html_default  # Library-derived ID
assert 'setAlignSpeed' in html_default           # Library-derived onchange JS
assert 'sd-align-auto-nav-toggle' in html_default
assert 'sd-align-audio-controls' in html_default
assert 'hx-post' not in html_default             # speed_url empty → no HTMX
assert 'setAlignSpeed(1.0)' not in html_default  # speed==1.0 → sync script is no-op

# Non-default speed + change_url: HTMX persistence wired, sync script fires, selected option matches
html_wired = to_xml(render_align_audio_controls(current_speed=2.0, auto_navigate=True, speed_url="/x/speed"))
assert 'hx-post="/x/speed"' in html_wired
assert 'hx-trigger="change"' in html_wired
assert 'setAlignSpeed(2.0)' in html_wired  # Initial sync script
assert re.search(r'<option[^>]*value="2.0"[^>]*selected', html_wired)

# OOB wrapper
html_oob = to_xml(render_align_audio_controls(auto_navigate=True, oob=True))
assert 'hx-swap-oob' in html_oob
assert 'sd-align-audio-controls' in html_oob

print('Alignment audio controls tests passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()